# 06. Building Your First Complete Agent

Welcome to the capstone of the beginner curriculum. We build **one complete, bounded, testable agent** using the defense-in-depth architecture typical of enterprise production systems.

**Scenario:** A customer escalates ticket `T-102`: *"I was charged twice for my subscription. Support has not resolved this. Please fix it."*

### Production Architecture Highlights
- **Multi-Tenant Data Isolation:** Domain fixtures and dispatchers enforce organization boundaries (`tenant_id`).
- **Cryptographic Human Approval Tokens:** High-consequence side effects require unexpired, cryptographically bound approvals from authorized managers.
- **Strict Error Taxonomy & Sanitization:** Explicit differentiation between `ValidationError`, `DomainValidationError`, `AuthorizationError`, and sanitized `INTERNAL_TOOL_ERROR`.
- **Separation of Proposal from Execution:** The LLM can only emit a `RefundProposal`; side effects can only execute after approval verification.

In [1]:
import json
import time
import os
import logging
import hashlib
from typing import Optional, Literal, Any, List, Set, Dict, Callable, Union
from enum import Enum
from pydantic import BaseModel, Field, ConfigDict, ValidationError

MODEL_NAME = 'gpt-4o-mini'
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('Course06')
print(f'Environment initialized. Configured model: {MODEL_NAME}')

Environment initialized. Configured model: gpt-4o-mini


## Part 1: Multi-Tenant Domain Fixtures & Explicit Error Types

All fixtures explicitly include `tenant_id` to ensure strict organizational data isolation. We define custom domain exceptions for clear separation of concerns.

In [2]:
class DomainValidationError(Exception):
    """Raised when business domain rules fail."""
    pass

class AuthorizationError(Exception):
    """Raised when role or tenant scope validation fails."""
    pass

class ToolExecutionError(Exception):
    """Raised when internal execution errors occur."""
    pass

DB = {
    'tickets': {
        'T-102': {'tenant_id': 'Northstar', 'customer_id': 'C-55', 'text': 'I was charged twice for my subscription. Please fix it.', 'status': 'open'},
        'T-999': {'tenant_id': 'AcmeCorp', 'customer_id': 'C-99', 'text': 'Acme cross-tenant ticket.', 'status': 'open'}
    },
    'transactions': {
        'C-55': [
            {'tenant_id': 'Northstar', 'tx_id': 'TX-901', 'amount_cents': 10000, 'date': '2026-08-01', 'note': 'initial subscription charge'},
            {'tenant_id': 'Northstar', 'tx_id': 'TX-902', 'amount_cents': 10000, 'date': '2026-08-01', 'note': 'system duplicate charge'}
        ],
        'C-99': [
            {'tenant_id': 'AcmeCorp', 'tx_id': 'TX-801', 'amount_cents': 5000, 'date': '2026-08-01', 'note': 'acme single charge'}
        ]
    },
    'refunds': []
}

print('Multi-tenant domain fixtures initialized.')

Multi-tenant domain fixtures initialized.


## Part 2: Application-Provided Trusted Context

The model does not provide its own identity. The application injects an `ExecutionContext` with actor ID, organization tenant, and roles.

In [3]:
class ExecutionContext(BaseModel):
    actor_id: str = Field(..., description='Unique identifier of the agent or operator')
    tenant_id: str = Field(..., description='Organization tenant boundary')
    roles: Set[str] = Field(default_factory=set, description='Assigned security permissions')
    request_id: str = Field(..., description='Correlation request ID')

ctx_northstar_agent = ExecutionContext(
    actor_id='agent_007',
    tenant_id='Northstar',
    roles={'support:read', 'billing:read', 'refund:issue'},
    request_id='REQ-101'
)

ctx_trainee_agent = ExecutionContext(
    actor_id='trainee_001',
    tenant_id='Northstar',
    roles={'support:read'},
    request_id='REQ-102'
)

ctx_acme_agent = ExecutionContext(
    actor_id='acme_user_1',
    tenant_id='AcmeCorp',
    roles={'support:read', 'billing:read', 'refund:issue'},
    request_id='REQ-103'
)

print(f'Trusted context configured: {ctx_northstar_agent}')

Trusted context configured: actor_id='agent_007' tenant_id='Northstar' roles={'support:read', 'refund:issue', 'billing:read'} request_id='REQ-101'


## Part 3: Read-Only Tool Contracts & Handlers

We define strongly typed Pydantic models with `extra='forbid'` to prevent parameter injection.

In [4]:
# Output Models
class TicketDetails(BaseModel):
    tenant_id: str
    customer_id: str
    text: str
    status: str

class Transaction(BaseModel):
    tenant_id: str
    tx_id: str
    amount_cents: int
    date: str
    note: Optional[str] = None

class TransactionList(BaseModel):
    transactions: List[Transaction] = Field(default_factory=list)

class RefundPolicy(BaseModel):
    policy_text: str

# Argument Models
class GetTicketArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    ticket_id: str = Field(..., description='The ticket identifier')

class GetCustomerArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    customer_id: str = Field(..., description='The customer identifier')

class GetRefundPolicyArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')

# Read Handlers
def get_ticket_details_impl(args: GetTicketArgs) -> TicketDetails:
    t_data = DB['tickets'][args.ticket_id]
    return TicketDetails(**t_data)

def get_recent_transactions_impl(args: GetCustomerArgs) -> TransactionList:
    tx_list = DB['transactions'][args.customer_id]
    return TransactionList(transactions=[Transaction(**t) for t in tx_list])

def get_refund_policy_impl(args: GetRefundPolicyArgs) -> RefundPolicy:
    return RefundPolicy(policy_text='Duplicate charges must be refunded in full. Consequential refunds require human manager approval.')

print('Read-only tool contracts and handlers initialized.')

Read-only tool contracts and handlers initialized.


## Part 4: Proposals & Time-Bound Approval Tokens

The model produces a `RefundProposal`. A human manager reviews and signs an `Approval` bound to a SHA-256 digest of the proposal and an `expires_at` timestamp.

In [5]:
class ApprovalDecision(str, Enum):
    APPROVE = 'approve'
    REJECT = 'reject'

class RefundProposal(BaseModel):
    customer_id: str
    transaction_id: str
    amount_cents: int
    reason: str

class Approval(BaseModel):
    proposal_digest: str = Field(..., description='SHA-256 hash of the specific proposal')
    approver_id: str = Field(..., description='Authorized manager ID')
    decision: ApprovalDecision
    expires_at: float = Field(..., description='Unix timestamp after which approval is invalid')

ALLOWED_APPROVERS: Set[str] = {'MGR-1', 'MGR-2', 'COMPLIANCE-LEAD'}

def hash_proposal(proposal: RefundProposal) -> str:
    return hashlib.sha256(proposal.model_dump_json().encode()).hexdigest()

print('Proposal and Approval structures initialized.')

Proposal and Approval structures initialized.


## Part 5: The Consequential Write Tool (Refund Execution)

This tool executes the side-effect. It includes `tenant_id` and an idempotency key.

In [6]:
class IssueRefundArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    tenant_id: str
    customer_id: str
    transaction_id: str
    amount_cents: int
    idempotency_key: str

class RefundResult(BaseModel):
    status: Literal['success', 'already_processed']
    amount_cents: int
    transaction_id: str

def _issue_refund_impl(args: IssueRefundArgs) -> RefundResult:
    # Idempotency check
    for r in DB['refunds']:
        if r['idempotency_key'] == args.idempotency_key:
            return RefundResult(status='already_processed', amount_cents=args.amount_cents, transaction_id=args.transaction_id)
    
    # Execute write
    DB['refunds'].append(args.model_dump())
    return RefundResult(status='success', amount_cents=args.amount_cents, transaction_id=args.transaction_id)

print('Write handler initialized.')

Write handler initialized.


## Part 6: Tool Registry & Sanitized Safe Dispatcher

The dispatcher validates schemas with `ValidationError`, checks actor roles and tenant scopes, validates domain rules, and sanitizes internal exceptions (`INTERNAL_TOOL_ERROR`).

In [7]:
class ToolDefinition(BaseModel):
    name: str
    effect: Literal['READ_ONLY', 'CONSEQUENTIAL_WRITE']
    required_permission: str
    input_model: type[BaseModel]
    result_model: type[BaseModel]
    func: Optional[Callable] = None

TOOL_REGISTRY = {
    'get_ticket_details': ToolDefinition(name='get_ticket_details', effect='READ_ONLY', required_permission='support:read', input_model=GetTicketArgs, result_model=TicketDetails, func=get_ticket_details_impl),
    'get_recent_transactions': ToolDefinition(name='get_recent_transactions', effect='READ_ONLY', required_permission='billing:read', input_model=GetCustomerArgs, result_model=TransactionList, func=get_recent_transactions_impl),
    'get_refund_policy': ToolDefinition(name='get_refund_policy', effect='READ_ONLY', required_permission='support:read', input_model=GetRefundPolicyArgs, result_model=RefundPolicy, func=get_refund_policy_impl),
    'issue_refund': ToolDefinition(name='issue_refund', effect='CONSEQUENTIAL_WRITE', required_permission='refund:issue', input_model=IssueRefundArgs, result_model=RefundResult, func=_issue_refund_impl),
}

class ToolError(BaseModel):
    error_type: Literal['SCHEMA_ERROR', 'DOMAIN_VALIDATION_ERROR', 'AUTHORIZATION_DENIED', 'INTERNAL_TOOL_ERROR', 'UNKNOWN_TOOL']
    error: str

def dispatch_tool(tool_name: str, args_dict: dict, ctx: ExecutionContext) -> Union[BaseModel, ToolError]:
    if tool_name not in TOOL_REGISTRY:
        return ToolError(error_type='UNKNOWN_TOOL', error=f"Unknown tool: '{tool_name}'")
        
    tdef = TOOL_REGISTRY[tool_name]
    
    # Role authorization check
    if tdef.required_permission not in ctx.roles:
        return ToolError(error_type='AUTHORIZATION_DENIED', error=f"Missing required role: {tdef.required_permission}")
        
    # Typed schema validation
    try:
        args_obj = tdef.input_model(**args_dict)
    except ValidationError as e:
        return ToolError(error_type='SCHEMA_ERROR', error=f"Schema validation error: {e.errors()[0]['msg']}")
    except Exception as e:
        logger.error(f'Unexpected argument processing error: {e}', exc_info=True)
        return ToolError(error_type='INTERNAL_TOOL_ERROR', error='INTERNAL_TOOL_ERROR: Could not parse arguments.')

    # Business & Tenant Scope Enforcement
    try:
        if tool_name == 'get_ticket_details':
            if args_obj.ticket_id not in DB['tickets']:
                raise DomainValidationError(f"Ticket '{args_obj.ticket_id}' not found.")
            t_data = DB['tickets'][args_obj.ticket_id]
            if t_data['tenant_id'] != ctx.tenant_id:
                raise AuthorizationError('Cross-tenant access denied: ticket belongs to another organization.')
            return tdef.func(args_obj)

        elif tool_name == 'get_recent_transactions':
            if args_obj.customer_id not in DB['transactions']:
                raise DomainValidationError(f"Customer '{args_obj.customer_id}' not found.")
            tx_list = DB['transactions'][args_obj.customer_id]
            for tx in tx_list:
                if tx['tenant_id'] != ctx.tenant_id:
                    raise AuthorizationError('Cross-tenant access denied: transaction belongs to another organization.')
            return tdef.func(args_obj)

        elif tool_name == 'get_refund_policy':
            return tdef.func(args_obj)

        elif tool_name == 'issue_refund':
            if args_obj.tenant_id != ctx.tenant_id:
                raise AuthorizationError('Cross-tenant refund denied: tenant parameter mismatch.')
            customer_txs = DB['transactions'].get(args_obj.customer_id, [])
            tx = next((t for t in customer_txs if t['tx_id'] == args_obj.transaction_id), None)
            if not tx:
                raise DomainValidationError('Transaction not found for this customer.')
            if tx['tenant_id'] != ctx.tenant_id:
                raise AuthorizationError('Cross-tenant refund denied: transaction belongs to another organization.')
            if tx['amount_cents'] != args_obj.amount_cents:
                raise DomainValidationError('Refund amount must match transaction amount strictly.')
            if 'duplicate' not in tx.get('note', '').lower():
                raise DomainValidationError('Transaction is not marked as a duplicate charge.')
            return tdef.func(args_obj)

    except AuthorizationError as e:
        return ToolError(error_type='AUTHORIZATION_DENIED', error=str(e))
    except DomainValidationError as e:
        return ToolError(error_type='DOMAIN_VALIDATION_ERROR', error=str(e))
    except Exception as e:
        # Sanitize internal exceptions
        logger.error(f'Internal unexpected error executing {tool_name}: {e}', exc_info=True)
        return ToolError(error_type='INTERNAL_TOOL_ERROR', error='INTERNAL_TOOL_ERROR: Operation failed safely.')

print('Safe dispatcher initialized with tenant isolation and error sanitization.')

Safe dispatcher initialized with tenant isolation and error sanitization.


## Part 7: Runtime State Machine & Decision Models

We define observable `AgentDecision` and immutable state tracking.

In [8]:
class ToolCall(BaseModel):
    id: str
    name: str
    arguments: Dict[str, Any] = Field(default_factory=dict)

class AgentDecision(BaseModel):
    decision_summary: str = Field(..., description='Observable action rationale')
    tool_calls: List[ToolCall] = Field(default_factory=list)
    proposal: Optional[RefundProposal] = None
    final_answer: Optional[str] = None

class AgentState(BaseModel):
    ticket_id: str
    steps: int = 0
    max_steps: int = 6
    history: List[Dict[str, Any]] = Field(default_factory=list)
    seen_actions: Set[str] = Field(default_factory=set)
    terminal_reason: Optional[Literal[
        'SUCCESS', 'NO_REFUND_NEEDED', 'INSUFFICIENT_EVIDENCE', 
        'AUTHORIZATION_DENIED', 'APPROVAL_REQUIRED', 'HUMAN_REJECTED', 
        'STEP_BUDGET_EXHAUSTED', 'NO_PROGRESS'
    ]] = None
    proposal: Optional[RefundProposal] = None
    approval: Optional[Approval] = None
    current_time: Optional[float] = None  # Configurable time for deterministic expiration tests

print('AgentState and AgentDecision models initialized.')

AgentState and AgentDecision models initialized.


## Part 8: The Evidence-Based Mock Decision Model

The model inspects historical observations to dynamically determine next steps.

In [9]:
class MockDecisionModel:
    def decide(self, state: AgentState) -> AgentDecision:
        history_str = json.dumps(state.history)
        
        # 1. Fetch ticket details if not yet observed
        if not any(h.get('tool') == 'get_ticket_details' for h in state.history):
            return AgentDecision(
                decision_summary=f'Retrieve details for ticket {state.ticket_id}',
                tool_calls=[ToolCall(id='tc_1', name='get_ticket_details', arguments={'ticket_id': state.ticket_id})]
            )
            
        # 2. Fetch customer transactions and refund policy
        if not any(h.get('tool') == 'get_recent_transactions' for h in state.history):
            return AgentDecision(
                decision_summary='Retrieve customer transactions and check refund policy',
                tool_calls=[
                    ToolCall(id='tc_2', name='get_recent_transactions', arguments={'customer_id': 'C-55'}),
                    ToolCall(id='tc_3', name='get_refund_policy', arguments={})
                ]
            )
            
        # 3. Analyze evidence: if duplicate charge confirmed, emit proposal
        if 'charged twice' in history_str and 'TX-902' in history_str:
            return AgentDecision(
                decision_summary='Duplicate transaction TX-902 identified. Submitting refund proposal for manager approval.',
                proposal=RefundProposal(
                    customer_id='C-55',
                    transaction_id='TX-902',
                    amount_cents=10000,
                    reason='Duplicate subscription charge confirmed in TX-902.'
                )
            )
            
        return AgentDecision(
            decision_summary='Investigation inconclusive. No duplicate charge identified.',
            final_answer='I investigated but could not find a duplicate charge.'
        )

print('MockDecisionModel initialized.')

MockDecisionModel initialized.


## Part 9: Core Bounded Execution Runtime

Handles autonomous execution loops, approval token verification, step budgets, and side-effect postconditions.

In [10]:
def run_agent(ctx: ExecutionContext, state: AgentState, model) -> AgentState:
    # --- Resumption with Manager Approval ---
    if state.proposal and state.approval:
        if state.approval.decision == ApprovalDecision.REJECT:
            state.terminal_reason = 'HUMAN_REJECTED'
            return state
            
        now = state.current_time if state.current_time is not None else time.time()
        
        # 1. Expiration Check
        if now > state.approval.expires_at:
            logger.warning('Approval token has expired.')
            state.terminal_reason = 'AUTHORIZATION_DENIED'
            return state
            
        # 2. Authorized Approver Identity Check
        if state.approval.approver_id not in ALLOWED_APPROVERS:
            logger.warning(f'Unauthorized approver ID: {state.approval.approver_id}')
            state.terminal_reason = 'AUTHORIZATION_DENIED'
            return state
            
        # 3. Cryptographic Proposal Digest Verification
        computed_digest = hash_proposal(state.proposal)
        if computed_digest != state.approval.proposal_digest:
            logger.warning('Proposal digest mismatch (proposal was mutated after signing).')
            state.terminal_reason = 'AUTHORIZATION_DENIED'
            return state
            
        idem_key = f'ref_{state.proposal.transaction_id}_{computed_digest[:8]}'
        refund_args = {
            'tenant_id': ctx.tenant_id,
            'customer_id': state.proposal.customer_id,
            'transaction_id': state.proposal.transaction_id,
            'amount_cents': state.proposal.amount_cents,
            'idempotency_key': idem_key
        }
        
        # Execute Consequential Write via Safe Dispatcher
        res = dispatch_tool('issue_refund', refund_args, ctx)
        state.history.append({'tool': 'issue_refund', 'result': res.model_dump()})
        
        if isinstance(res, RefundResult):
            # Verify postcondition in database
            if any(r['idempotency_key'] == idem_key for r in DB['refunds']):
                state.terminal_reason = 'SUCCESS'
            else:
                state.terminal_reason = 'INSUFFICIENT_EVIDENCE'
        else:
            state.terminal_reason = 'AUTHORIZATION_DENIED'
        return state

    # --- Standard Execution Loop ---
    while state.terminal_reason is None:
        if state.steps >= state.max_steps:
            state.terminal_reason = 'STEP_BUDGET_EXHAUSTED'
            break
            
        state.steps += 1
        decision: AgentDecision = model.decide(state)
        print(f'Step {state.steps} | Action rationale: {decision.decision_summary}')
        
        if decision.final_answer:
            state.terminal_reason = 'NO_REFUND_NEEDED'
            break
            
        if decision.proposal:
            state.proposal = decision.proposal
            state.terminal_reason = 'APPROVAL_REQUIRED'
            break
            
        for tc in decision.tool_calls:
            fingerprint = f'{tc.name}:{json.dumps(tc.arguments, sort_keys=True)}'
            if fingerprint in state.seen_actions:
                state.terminal_reason = 'NO_PROGRESS'
                break
            state.seen_actions.add(fingerprint)
            
            # Block unauthorized attempts to call write tools directly
            if TOOL_REGISTRY[tc.name].effect != 'READ_ONLY':
                state.terminal_reason = 'AUTHORIZATION_DENIED'
                break
                
            res = dispatch_tool(tc.name, tc.arguments, ctx)
            state.history.append({'tool': tc.name, 'result': res.model_dump()})
            
    return state

print('Execution runtime initialized.')

Execution runtime initialized.


## Part 10: Exhaustive Test Suite & Invariant Verification

We execute test cases verifying the full defense-in-depth security model.

In [11]:
now = time.time()
model = MockDecisionModel()

print('=== 1. Test: Happy Path (Proposal -> Approval -> Refund) ===')
st1 = run_agent(ctx_northstar_agent, AgentState(ticket_id='T-102', current_time=now), model)
assert st1.terminal_reason == 'APPROVAL_REQUIRED'
assert st1.proposal is not None

valid_digest = hash_proposal(st1.proposal)
st1.approval = Approval(
    proposal_digest=valid_digest,
    approver_id='MGR-1',
    decision=ApprovalDecision.APPROVE,
    expires_at=now + 300
)
st1 = run_agent(ctx_northstar_agent, st1, model)
assert st1.terminal_reason == 'SUCCESS'
print('Happy path passed.')

print('\n=== 2. Test: Idempotency (Duplicate Execution Guard) ===')
st1.terminal_reason = None
st1 = run_agent(ctx_northstar_agent, st1, model)
assert st1.terminal_reason == 'SUCCESS'
assert len([r for r in DB['refunds'] if r['transaction_id'] == 'TX-902']) == 1
print('Idempotency passed.')

print('\n=== 3. Test: Cross-Tenant Ticket Access Blocked ===')
st_cross_ticket = run_agent(ctx_northstar_agent, AgentState(ticket_id='T-999', current_time=now), model)
assert any(h.get('result', {}).get('error_type') == 'AUTHORIZATION_DENIED' for h in st_cross_ticket.history)
print('Cross-tenant ticket block passed.')

print('\n=== 4. Test: Cross-Tenant Refund Blocked ===')
st_cross_refund = AgentState(
    ticket_id='T-102',
    proposal=RefundProposal(customer_id='C-99', transaction_id='TX-801', amount_cents=5000, reason='Attempting cross-tenant refund'),
    current_time=now
)
st_cross_refund.approval = Approval(
    proposal_digest=hash_proposal(st_cross_refund.proposal),
    approver_id='MGR-1',
    decision=ApprovalDecision.APPROVE,
    expires_at=now + 300
)
st_cross_refund = run_agent(ctx_northstar_agent, st_cross_refund, model)
assert st_cross_refund.terminal_reason == 'AUTHORIZATION_DENIED'
print('Cross-tenant refund block passed.')

print('\n=== 5. Test: Expired Approval Blocked ===')
st_expired = AgentState(
    ticket_id='T-102',
    proposal=RefundProposal(customer_id='C-55', transaction_id='TX-902', amount_cents=10000, reason='Expired test'),
    current_time=now
)
st_expired.approval = Approval(
    proposal_digest=hash_proposal(st_expired.proposal),
    approver_id='MGR-1',
    decision=ApprovalDecision.APPROVE,
    expires_at=now - 10  # Expired 10 seconds ago
)
st_expired = run_agent(ctx_northstar_agent, st_expired, model)
assert st_expired.terminal_reason == 'AUTHORIZATION_DENIED'
print('Expired approval rejection passed.')

print('\n=== 6. Test: Mutated Proposal / Digest Mismatch Blocked ===')
st_mutated = AgentState(
    ticket_id='T-102',
    proposal=RefundProposal(customer_id='C-55', transaction_id='TX-902', amount_cents=10000, reason='Original proposal'),
    current_time=now
)
st_mutated.approval = Approval(
    proposal_digest='forged_digest_hex_12345',
    approver_id='MGR-1',
    decision=ApprovalDecision.APPROVE,
    expires_at=now + 300
)
st_mutated = run_agent(ctx_northstar_agent, st_mutated, model)
assert st_mutated.terminal_reason == 'AUTHORIZATION_DENIED'
print('Digest mismatch rejection passed.')

print('\n=== 7. Test: Unauthorized Approver Blocked ===')
st_bad_approver = AgentState(
    ticket_id='T-102',
    proposal=RefundProposal(customer_id='C-55', transaction_id='TX-902', amount_cents=10000, reason='Unauthorized approver test'),
    current_time=now
)
st_bad_approver.approval = Approval(
    proposal_digest=hash_proposal(st_bad_approver.proposal),
    approver_id='HACKER_99',
    decision=ApprovalDecision.APPROVE,
    expires_at=now + 300
)
st_bad_approver = run_agent(ctx_northstar_agent, st_bad_approver, model)
assert st_bad_approver.terminal_reason == 'AUTHORIZATION_DENIED'
print('Unauthorized approver rejection passed.')

print('\nAll 7 security and invariant tests passed successfully!')

=== 1. Test: Happy Path (Proposal -> Approval -> Refund) ===
Step 1 | Action rationale: Retrieve details for ticket T-102
Step 2 | Action rationale: Retrieve customer transactions and check refund policy
Step 3 | Action rationale: Duplicate transaction TX-902 identified. Submitting refund proposal for manager approval.
Happy path passed.

=== 2. Test: Idempotency (Duplicate Execution Guard) ===
Idempotency passed.

=== 3. Test: Cross-Tenant Ticket Access Blocked ===
Step 1 | Action rationale: Retrieve details for ticket T-999
Step 2 | Action rationale: Retrieve customer transactions and check refund policy
Step 3 | Action rationale: Investigation inconclusive. No duplicate charge identified.
Cross-tenant ticket block passed.

=== 4. Test: Cross-Tenant Refund Blocked ===
Cross-tenant refund block passed.

=== 5. Test: Expired Approval Blocked ===
Expired approval rejection passed.

=== 6. Test: Mutated Proposal / Digest Mismatch Blocked ===
Digest mismatch rejection passed.

=== 7. Test

## Part 11: Optional Real OpenAI Implementation

*(Optional)* When `OPENAI_API_KEY` is present, we run the **exact same architecture** with `gpt-4o-mini`.

- The model is provided only `propose_refund` (not `issue_refund`), guaranteeing that side effects cannot be initiated directly.
- Model tool proposals are mapped into internal `ToolCall` and `RefundProposal` models and executed through `dispatch_tool`.

In [12]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY detected in environment. Skipping live OpenAI API run.')
else:
    from openai import OpenAI
    
    class OpenAIDecisionModel:
        def __init__(self):
            self.client = OpenAI(api_key=api_key)
            self.conversation_messages = [
                {'role': 'system', 'content': 'You are a customer support agent. Investigate duplicate subscription charges. If a duplicate charge is verified, propose a refund with propose_refund. Otherwise output a final explanation.'}
            ]
            self.tools = [
                {'type': 'function', 'function': {'name': 'get_ticket_details', 'description': 'Retrieve ticket details', 'parameters': GetTicketArgs.model_json_schema()}},
                {'type': 'function', 'function': {'name': 'get_recent_transactions', 'description': 'Retrieve customer transactions', 'parameters': GetCustomerArgs.model_json_schema()}},
                {'type': 'function', 'function': {'name': 'get_refund_policy', 'description': 'Retrieve refund policy rules', 'parameters': GetRefundPolicyArgs.model_json_schema()}},
                {'type': 'function', 'function': {'name': 'propose_refund', 'description': 'Submit a refund proposal for manager review', 'parameters': RefundProposal.model_json_schema()}}
            ]

        def decide(self, state: AgentState) -> AgentDecision:
            if len(self.conversation_messages) == 1:
                self.conversation_messages.append({'role': 'user', 'content': f'Please investigate ticket {state.ticket_id} regarding duplicate charges.'})

            response = self.client.chat.completions.create(model=MODEL_NAME, messages=self.conversation_messages, tools=self.tools)
            msg = response.choices[0].message
            self.conversation_messages.append(msg)

            if not msg.tool_calls:
                return AgentDecision(decision_summary='Direct answer generated by model.', final_answer=msg.content or '')

            tcs = []
            for tc in msg.tool_calls:
                try:
                    args = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    args = {}

                if tc.function.name == 'propose_refund':
                    return AgentDecision(
                        decision_summary='Model verified duplicate charge and emitted refund proposal.',
                        proposal=RefundProposal(**args)
                    )

                tcs.append(ToolCall(id=tc.id, name=tc.function.name, arguments=args))

            return AgentDecision(
                decision_summary=f'Model proposing tools: {[t.name for t in tcs]}',
                tool_calls=tcs
            )

    print(f'\n--- Running Live OpenAI Agent ({MODEL_NAME}) ---')
    live_state = AgentState(ticket_id='T-102')
    live_model = OpenAIDecisionModel()
    live_state = run_agent(ctx_northstar_agent, live_state, live_model)
    print('Live Agent terminal reason:', live_state.terminal_reason)
    if live_state.proposal:
        print('Live Proposal Emitted:', live_state.proposal.model_dump_json())

No OPENAI_API_KEY detected in environment. Skipping live OpenAI API run.
